# 01 — Unique events: Variable 1nt vs Strict

This notebook identifies which splicing events are detected by SUPPA
in variable 1nt mode but NOT in strict mode, at the generateEvents level.

This comparison is purely structural — it does not involve expression
or statistical testing. We are comparing the event catalogues produced
by generateEvents with different boundary parameters.

**Events analysed:** A3, A5, RI  
**Comparison:** Strict IOE vs Variable 1nt IOE

In [158]:
import pandas as pd
import os
import matplotlib.pyplot as plt

# IOE file paths
IOE_STRICT = "/Users/gricey/Desktop/Internship/data/output_strict/events"
IOE_VAR1   = "/Users/gricey/Desktop/Internship/data/output_variable_1nt/events"

# Events of interest
EVENTS = ["A3", "A5", "RI"]

# IOE filename suffixes
SUFFIX_STRICT = "strict"
SUFFIX_VAR1   = "variable_1"

# Output directory
OUTPUT_DIR = "/Users/gricey/Desktop/Internship/data/ioe_diff"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Helper functions ---

def style_table(df, caption=""):
    """Apply consistent blue-header styling to a DataFrame for display."""
    styled = df.style\
        .set_properties(**{
            "font-size": "12px",
            "font-weight": "bold",
            "border": "1px solid #ddd",
            "padding": "6px 12px",
            "text-align": "center"
        })\
        .set_table_styles([
            {"selector": "th", "props": [
                ("background-color", "#2196F3"),
                ("color", "white"),
                ("font-weight", "bold"),
                ("padding", "6px 12px"),
                ("text-align", "center")
            ]},
            {"selector": "tr:nth-child(even)", "props": [
                ("background-color", "#f2f2f2")
            ]},
        ])\
        .hide(axis="index")
    if caption:
        styled = styled.set_caption(caption)
    return styled

def export_table_png(df, filepath, title="", figsize=None):
    """Export a DataFrame as a styled PNG image."""
    if figsize is None:
        figsize = (len(df.columns) * 1.8, len(df) * 0.6 + 0.8)
    fig, ax = plt.subplots(figsize=figsize)
    ax.axis("off")
    table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        cellLoc="center",
        loc="center"
    )
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1, 1.6)
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_facecolor("#2196F3")
            cell.set_text_props(color="white", fontweight="bold")
        elif row % 2 == 0:
            cell.set_facecolor("#f2f2f2")
        else:
            cell.set_facecolor("white")
        cell.set_edgecolor("#dddddd")
    if title:
        ax.set_title(title, fontsize=12, pad=10)
    plt.tight_layout()
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved -> {filepath}")

print("Setup done!")

Setup done!


## The mental model - 3 levels of detail
### level 1: The event (one row in the IOE file)
(suppa_env) gricey@L108422 ~ % head -5 ~/Desktop/Internship/data/output_strict/events/events_A3_strict.ioe

gene_id                     event_id                                    
ENSMUSG00000033021       ENSMUSG00000033021;A3:1:75412655-75413422:75412655-75413428:+

ENSMUSG00000033021       ENSMUSG00000033021;A3:1:75416048-75417056:75416048-75417132:+

ENSMUSG00000033021       ENSMUSG00000033021;A3:1:75412956-75413422:75412956-75413428:+

**ENSMUSG00000033021** appers 3 times. That's one gene with **3 different A3 splicing events**. 3 different places in the gene where the alternative 3' splice site can occur.

So:
- 1 gene can have multiple events 
- 1 event is one specific splicing decision at one specific location


### level 2: The gene
The unique gene file **A3_var1_unique_genes.txt** just list gene ID's:

ENSMUSG00000000085
ENSMUSG00000000088
...


These are genes that have **at least one A3 event detected in variable 1nt mode** but **zero A3 events in strict mode.** The gene wasn't detected at all in strict, not even once.



## Step 1 — Load IOE files

We load the IOE files produced by generateEvents for both strict and
variable 1nt modes. Each file contains one row per detected splicing event.

In [159]:
ioe_data = {}

for event in EVENTS: # EVENTS = ["A3", "A5", "RI"] 
    strict_path = f"{IOE_STRICT}/events_{event}_{SUFFIX_STRICT}.ioe"
    var1_path   = f"{IOE_VAR1}/events_{event}_{SUFFIX_VAR1}.ioe"

    # Load IOE data for this event type
    ioe_data[event] = {
        "strict": pd.read_csv(strict_path, sep="\t"),
        "var1":   pd.read_csv(var1_path,   sep="\t"),
    }

    print(f"{event} — strict: {len(ioe_data[event]['strict'])} events, "
          f"variable 1nt: {len(ioe_data[event]['var1'])} events")

A3 — strict: 11163 events, variable 1nt: 22931 events
A5 — strict: 9629 events, variable 1nt: 19455 events
RI — strict: 5315 events, variable 1nt: 17336 events


## Step 2 — Count summary

Simple count of how many events each mode detects per event type,
and the raw difference V1 - S.

In [160]:
rows = []
for event in EVENTS:
    n_strict = len(ioe_data[event]["strict"])
    n_var1   = len(ioe_data[event]["var1"])
    rows.append({
        "Event":          event,
        "Strict":         n_strict,
        "Variable 1nt":   n_var1,
        "V1 - S":         n_var1 - n_strict,
    })

df_counts = pd.DataFrame(rows)
display(style_table(df_counts, "Table 1 — IOE event counts: Strict vs Variable 1nt"))
export_table_png(df_counts,
                 "../../figures/plots/table_ioe_counts.png",
                 title="IOE event counts — Strict vs Variable 1nt")

Event,Strict,Variable 1nt,V1 - S
A3,11163,22931,11768
A5,9629,19455,9826
RI,5315,17336,12021


Saved -> ../../figures/plots/table_ioe_counts.png


## Step 3 — Find unique events in Variable 1nt

We extract the gene IDs from each IOE file and find which genes
are present in variable 1nt but completely absent in strict.

Note: we compare at the gene level (ENSMUSG ID) because strict and
variable mode use different coordinate systems for the event boundaries.

In [161]:
# Extract the set of unique gene IDs from an IOE dataframe -----------------------------------------
def get_gene_ids(ioe_df):
    return set(ioe_df["gene_id"].unique()) # Extract unique gene IDs from the "gene_id" column of the IOE dataframe

unique_results = {} # stores results per event type
rows_unique = []    # stores summary rows for the table

# Get the set of gene IDs detected in each mode for each event type --------------------------------
for event in EVENTS: # loops 3 times: EVENTS = ["A3", "A5", "RI"]
    strict_genes = get_gene_ids(ioe_data[event]["strict"])
    var1_genes   = get_gene_ids(ioe_data[event]["var1"])

    # Set difference: genes in Variable 1nt but NOT in Strict
    unique_genes = var1_genes - strict_genes # Genes in Variable 1nt but not in Strict


    print(f"Event: {unique_genes }")
    # Filter the variable IOE dataframe to keep only the unique genes
    unique_results[event] = {
        "unique_genes": unique_genes,
        "ioe_unique":   ioe_data[event]["var1"][
            ioe_data[event]["var1"]["gene_id"].isin(unique_genes)
        ]
    }

    # Append summary row for this event type
    rows_unique.append({
        "Event":            event,
        "Genes in Strict":     len(strict_genes),
        "Genes in Variable 1nt": len(var1_genes),
        "different genes": len(unique_genes),
    })

    print(f"{event} — strict: {len(strict_genes)} genes, "
        f"variable 1nt: {len(var1_genes)} genes, "
        f"unique to variable: {len(unique_genes)} genes")

# Display summary table of unique genes per event type --------------------------------
df_unique = pd.DataFrame(rows_unique)
display(style_table(df_unique, "Table 2 — Genes unique to Variable 1nt vs Strict"))

export_table_png(df_unique,
                 "../../figures/plots/table_unique_genes.png",
                 "Genes unique to Variable 1nt vs Strict")




Event: {'ENSMUSG00000020290', 'ENSMUSG00000020105', 'ENSMUSG00000040219', 'ENSMUSG00000036913', 'ENSMUSG00000030872', 'ENSMUSG00000028969', 'ENSMUSG00000035969', 'ENSMUSG00000029581', 'ENSMUSG00000017119', 'ENSMUSG00000109561', 'ENSMUSG00000041991', 'ENSMUSG00000024576', 'ENSMUSG00000038260', 'ENSMUSG00000022040', 'ENSMUSG00000030016', 'ENSMUSG00000097779', 'ENSMUSG00000040857', 'ENSMUSG00000024944', 'ENSMUSG00000033392', 'ENSMUSG00000085399', 'ENSMUSG00000024369', 'ENSMUSG00000087241', 'ENSMUSG00000002984', 'ENSMUSG00000052749', 'ENSMUSG00000030659', 'ENSMUSG00000027834', 'ENSMUSG00000086152', 'ENSMUSG00000061111', 'ENSMUSG00000032508', 'ENSMUSG00000112198', 'ENSMUSG00000022952', 'ENSMUSG00000027867', 'ENSMUSG00000031241', 'ENSMUSG00000013155', 'ENSMUSG00000033105', 'ENSMUSG00000076688', 'ENSMUSG00000028716', 'ENSMUSG00000085601', 'ENSMUSG00000071653', 'ENSMUSG00000085620', 'ENSMUSG00000040811', 'ENSMUSG00000002524', 'ENSMUSG00000020680', 'ENSMUSG00000040181', 'ENSMUSG00000024334', 'E

Event,Genes in Strict,Genes in Variable 1nt,different genes
A3,6684,10107,3423
A5,5905,9002,3097
RI,3163,7568,4405


Saved -> ../../figures/plots/table_unique_genes.png


## Step 4 — Export unique events to IOE files

We save the IOE entries for the unique genes to new files.
These are the files to load into IGV for visualisation.

In [162]:
for event in EVENTS:
    ioe_out  = f"{OUTPUT_DIR}/{event}_var1_unique.ioe"
    gene_out = f"{OUTPUT_DIR}/{event}_var1_unique_genes.txt"

    # Save IOE file
    unique_results[event]["ioe_unique"].to_csv(ioe_out, sep="\t", index=False)

    # Save gene list
    with open(gene_out, "w") as f:
        for gene in sorted(unique_results[event]["unique_genes"]):
            f.write(gene + "\n")

    print(f"{event} -> {len(unique_results[event]['ioe_unique'])} IOE entries, "
          f"{len(unique_results[event]['unique_genes'])} unique genes")

A3 -> 5282 IOE entries, 3423 unique genes
A5 -> 4897 IOE entries, 3097 unique genes
RI -> 7979 IOE entries, 4405 unique genes


## Step 5 — Cross-check: consistency across steps

We verify that the numbers from Steps 2, 3 and 4 are consistent.

- **Step 2** counts raw IOE events (one row per splicing event)
- **Step 3** counts unique genes (one gene can have multiple events)
- **Step 4** counts IOE entries and genes exported to file

Expected relationships:
- Step 3 unique genes == Step 4 unique genes (same set, just exported)
- Step 2 V1-S >= Step 3 unique genes (more events than genes, since one gene can have multiple events)
- Step 4 IOE entries <= Step 2 V1-S (IOE entries for unique genes only)

In [163]:
# Step 2 data — raw IOE event counts
step2 = {
    "A3": {"strict": 11163, "var1": 22931, "diff": 11768},
    "A5": {"strict": 9629,  "var1": 19455, "diff": 9826},
    "RI": {"strict": 5315,  "var1": 17336, "diff": 12021},
}

rows_check = []

for event in EVENTS:
    # Step 2 values
    # Number of extra IOE events that variable detected vs strict
    s2_diff = step2[event]["diff"]          # extra IOE events in variable

    # Step 3 values — from unique_results computed earlier
    # Number of genes in variable but not strict
    s3_unique_genes = len(unique_results[event]["unique_genes"])  # unique genes

    # Step 4 values — from unique_results computed earlier
    # Number of IOE rows (spliced events) belonging to those unique genes
    s4_ioe_entries  = len(unique_results[event]["ioe_unique"])    # IOE entries exported

    # Consistency checks
    events_ge_genes = s2_diff >= s3_unique_genes                  # should always be True
    ioe_le_diff     = s4_ioe_entries <= s2_diff                   # should always be True

    rows_check.append({
        # The event type (A3, A5, RI)
        "Event type":                          event,
        
        # Total extra IOE events in variable vs strict
        # (one gene can contribute multiple events here)
        "Extra splicing events in variable":   s2_diff,
        
        # How many GENES are exclusively in variable (never seen in strict)
        "Genes only in variable":              s3_unique_genes,
        
        # How many IOE rows those unique genes represent
        # (multiple events per gene, so always >= unique genes)
        "IOE rows from unique genes":          s4_ioe_entries,
        
        # Quick sanity: IOE rows >= unique genes (one gene = multiple events)
        "IOE rows >= unique genes":            "ok" if events_ge_genes else "check!",
    })

    print(f"{event}:")
    print(f"  Extra splicing events in variable (V1 - Strict):  {s2_diff}")
    print(f"  Genes exclusively in variable (never in strict):  {s3_unique_genes}")
    print(f"  IOE rows belonging to those unique genes:         {s4_ioe_entries}")
    print(f"  Ratio events/genes: {s2_diff / s3_unique_genes:.1f} events per gene on average")
    print()

# Summary table
df_check = pd.DataFrame(rows_check)
display(style_table(df_check, "Cross-check: unique genes and their splicing events"))

export_table_png(
    df_check,
    "../../figures/plots/table_crosscheck.png",
    "Cross-check: unique genes and their splicing events",
    figsize=(18, 3)
)

A3:
  Extra splicing events in variable (V1 - Strict):  11768
  Genes exclusively in variable (never in strict):  3423
  IOE rows belonging to those unique genes:         5282
  Ratio events/genes: 3.4 events per gene on average

A5:
  Extra splicing events in variable (V1 - Strict):  9826
  Genes exclusively in variable (never in strict):  3097
  IOE rows belonging to those unique genes:         4897
  Ratio events/genes: 3.2 events per gene on average

RI:
  Extra splicing events in variable (V1 - Strict):  12021
  Genes exclusively in variable (never in strict):  4405
  IOE rows belonging to those unique genes:         7979
  Ratio events/genes: 2.7 events per gene on average



Event type,Extra splicing events in variable,Genes only in variable,IOE rows from unique genes,IOE rows >= unique genes
A3,11768,3423,5282,ok
A5,9826,3097,4897,ok
RI,12021,4405,7979,ok


Saved -> ../../figures/plots/table_crosscheck.png
